<a href="https://colab.research.google.com/github/aishanikar9/BWSI_Operations_Team/blob/main/ResnetModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

In [4]:
!wget -O RescueNet-classification-train.csv https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-train.csv
!wget -O RescueNet-classification-val.csv https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-val.csv
!wget -O RescueNet-classification-test.csv https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-test.csv

--2026-07-28 15:15:20--  https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 46761 (46K) [text/plain]
Saving to: ‘RescueNet-classification-train.csv’

RescueNet-classific 100%[===================>]  45.67K  --.-KB/s    in 0.006s  

2026-07-28 15:15:20 (7.31 MB/s) - ‘RescueNet-classification-train.csv’ saved [46761/46761]

--2026-07-28 15:15:20--  https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-val.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443.

In [5]:
import pandas as pd

train_df = pd.read_csv("RescueNet-classification-train.csv")
val_df = pd.read_csv("RescueNet-classification-val.csv")
test_df = pd.read_csv("RescueNet-classification-test.csv")

print(train_df.head())
print(val_df.head())
print(test_df.head())

    Image_ID  Neighborhood_ID
0  10930.jpg                2
1  11351.jpg                1
2  14254.jpg                0
3  15013.jpg                1
4  12373.jpg                1
    Image_ID  Neighborhood_ID
0  11251.jpg                2
1  11122.jpg                2
2  10997.jpg                2
3  11005.jpg                2
4  10925.jpg                2
    Image_ID  Neighborhood_ID
0  13363.jpg                1
1  11038.jpg                2
2  11965.jpg                2
3  13702.jpg                1
4  14376.jpg                1


In [6]:
net = torchvision.models.resnet50(weights = 'IMAGENET1K_V2')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 89.7MB/s]


In [7]:
net.fc = nn.Linear(2048, 3)
net = nn.DataParallel(net)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net.to(device)
print("ResNet ready")

ResNet ready


In [8]:
count = train_df["Neighborhood_ID"].value_counts().sort_index()
class_weights = torch.tensor((count.sum() / (len(count)* count)).values, dtype=torch.float).to(device)

#weighing different classes

In [34]:
for parameters in net.parameters():
  parameters.requires_grad = False
for parameters in net.module.fc.parameters():
  parameters.requires_grad = True

In [35]:
# Loss function, optimizer, scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, net.parameters()), lr=1e-3)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, 0.9)

In [36]:
#transform
flip = transforms.RandomHorizontalFlip(p=0.5) # some probability of flipping
scale = transforms.Resize(1024) # resize to fixed size
affine = transforms.RandomAffine(degrees=20, translate=(0.15, 0.15), scale=(0.8,1.6)) # affine transform
jitter = transforms.ColorJitter(brightness=0.4, contrast=0.3, saturation=0.3, hue=0.02) # add color/jitter noise
perspective = transforms.RandomPerspective(0.3, 0.5) # random perspective changes
c_crop = transforms.CenterCrop(512) # crop center 512x512 pixels

train_tf = transforms.Compose([scale,affine,perspective, jitter, c_crop, flip, transforms.ToTensor()])
evaluation_tf = transforms.Compose([scale,affine,perspective, jitter, c_crop, flip, transforms.ToTensor()])


In [37]:
#equivalent to class LadiDataset(Dataset):
class RescueNetDataset(Dataset):
    def __init__(self, label_csv, image_folder, transform = None):
        self.label_data_df = pd.read_csv(label_csv)
        self.image_folder = pathlib.Path(image_folder)
        self.transform = transform

    def __len__(self):
        return len(self.label_data_df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        ## Load images from the local dataset directory
        image_name = str(self.label_data_df.iloc[idx]["Image_ID"])
        local_path = self.image_folder / image_name
        image = Image.open(local_path).convert("RGB")
        label = torch.tensor(int(self.label_data_df.iloc[idx]["Neighborhood_ID"]), dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        example = {'image': image,
                   'image_name': image_name,
                   'label': label}

        return example

In [38]:
path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")
print("downloaded to:", path)


org = list(pathlib.Path(path).rglob("train-org-img"))
print("train-org-img:", org)

Using Colab cache for faster access to the 'rescuenet' dataset.
downloaded to: /kaggle/input/rescuenet
train-org-img: [PosixPath('/kaggle/input/rescuenet/RescueNet/train/train-org-img')]


In [39]:
train_image_directory = list(pathlib.Path(path).rglob("train-org-img"))[0]
val_image_directory   = list(pathlib.Path(path).rglob("val-org-img"))[0]

In [40]:
train_dataset = RescueNetDataset(label_csv= "RescueNet-classification-train.csv",
                                 image_folder = train_image_directory,
                                 transform = train_tf
)

val_dataset = RescueNetDataset(label_csv= "RescueNet-classification-val.csv",
                                 image_folder = train_image_directory,
                                 transform = evaluation_tf
)


In [41]:
#train_dataset and val_dataset will come from the RescueNetDataset() class
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=2)

In [42]:
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import classification_report, f1_score
import seaborn as sns

In [43]:
def evaluate_and_collect(net, test_loader, device):
    net.eval()
    all_preds = []
    all_targets = []
    use_amp = torch.cuda.is_available()

    with torch.no_grad():
        for data in test_loader:
            inputs = data['image'].to(device)
            labels = data['label'].to(device)
            with torch.amp.autocast(device_type='cuda' if use_amp else 'cpu', enabled=use_amp):
                outputs = net(inputs)
            _, predicted = torch.max(outputs, 1)
            all_preds.append(predicted.cpu().numpy())
            all_targets.append(labels.cpu().numpy())

    return np.concatenate(all_targets, axis=0), np.concatenate(all_preds, axis=0)

def show_and_save_confusion_matrix(y_true, y_pred, categories, epoch, logs_path):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)

    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix - Epoch {epoch}')
    plt.tight_layout()

    cm_folder = logs_path / 'confusion_matrices'
    cm_folder.mkdir(parents=True, exist_ok=True)
    plt.savefig(cm_folder / f'cm_epoch_{str(epoch).zfill(3)}.png', dpi=300)

    plt.show()
    plt.close()

In [44]:
#train

def train_model(net, train_loader, test_loader, criterion, optimizer, scheduler, logs_path, model_name,
                starting_epoch=0, additional_epochs=30, print_every_num_batches=50):
  model_name_base = f'Resnet50-{model_name}' + '.ep{}.pth'
  writer = SummaryWriter(logs_path)
  checkpoints_path = logs_path/'checkpoints'
  checkpoints_path.mkdir(parents=True, exist_ok=True)

  if starting_epoch > 0:
    load_path = checkpoints_path / model_name_base.format(str(starting_epoch).zfill(3))
    net.load_state_dict(torch.load(load_path, map_location=device))

  #adding mixed precision, less memory, higher speed
  use_amp = torch.cuda.is_available()
  scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

  #tracking the best model instead of just the first
  best_value_f1 = -1.0

  for epoch in range(starting_epoch, starting_epoch+additional_epochs):
    net.train()
    running_loss = 0.0
    running_epoch_loss = 0.0

    for i, data in enumerate(train_loader,0):
      inputs = data['image'].to(device)
      labels = data['label'].to(device)

      optimizer.zero_grad()

      #temporarily cuts bits in half, doubling run time and halfing GPU memory,
      with torch.autocast(device_type='cuda' if use_amp else 'cpu', enabled=use_amp):
        outputs = net(inputs)
        loss = criterion(outputs, labels)

      #multiplies loss when it get's really small so the model keeps learning, reversed by .backward() before weights update tho
      scaler.scale(loss).backward()

      #unscale, then update the weights, checks for infinite and NAN first tho
      scaler.step(optimizer)

      #update the scale amount before the next loop
      scaler.update()

      running_loss += loss.item()
      running_epoch_loss += loss.item()

      if (i + 1) % print_every_num_batches == 0:
        print(f'[epoch {epoch+1}, batch {i+1}] average loss: {running_loss/print_every_num_batches:.4f}')
        running_loss = 0.0

    categories = ["Superficial", "Medium", "Major"]

    average_epoch_loss = running_epoch_loss/(i+1)
    writer.add_scalar('Loss/epoch_avg/train', average_epoch_loss, epoch)
    print(f'[epoch {epoch+1}] average training epoch loss: {average_epoch_loss}')
    writer.add_scalar('LR/rate', scheduler.get_last_lr()[0], epoch)
    print(f'[epoch {epoch+1}] average training loss: {average_epoch_loss:.4f}')
    scheduler.step()

    y_test, predicted_y = evaluate_and_collect(net, test_loader, device)
    epoch_f1 = f1_score(y_test, predicted_y, average='macro')
    print(f'[epoch {epoch+1}] Validation F1 Score: {epoch_f1:.4f}')
    writer.add_scalar('Metric/F1_Score', epoch_f1, epoch)

    if epoch_f1 > best_value_f1:
        best_value_f1 = epoch_f1
        torch.save(net.state_dict(), checkpoints_path / f"BEST-{model_name_base.format(str(epoch+1).zfill(3))}")
        print(f"--> New best model saved with F1: {best_value_f1:.4f}")
        show_and_save_confusion_matrix(y_test, predicted_y, categories, f"BEST_ep{epoch+1}", logs_path)

    if (epoch + 1) % 5 == 0:
        print(classification_report(y_test, predicted_y))
        show_and_save_confusion_matrix(y_test, predicted_y, categories, epoch + 1, logs_path)

    epoch_string = str(epoch + 1).zfill(3)
    torch.save(net.state_dict(), checkpoints_path / model_name_base.format(epoch_string))


    # net.eval()
    # val_running = 0.0

    # with torch.no_grad(): #basically tells it to stop doing gradient computation
    #   for i, data in enumerate(test_loader, 0):
    #     inputs = data['image'].to(device)
    #     labels = data['label'].to(device)
    #     outputs = net(inputs)

    #     loss = criterion(outputs, labels)
    #     val_running += loss.item()

    #   val_loss = val_running / (i + 1)
  writer.close()

In [ ]:
logs_path = pathlib.Path("/content/drive/MyDrive/RescueNet_outputs")
logs_path.mkdir(parents=True, exist_ok=True)

train_model(net, train_loader, val_loader, criterion, optimizer, scheduler,
            logs_path=logs_path, model_name='damage_model',
            starting_epoch=0, additional_epochs=30, print_every_num_batches=50)